# Rebalancing Positions

**PyBroker** allows you to simulate portfolio rebalancing by adjusting your asset allocation to match a desired target. This notebook also demonstrates how to rebalance using [portfolio optimization](https://en.wikipedia.org/wiki/Portfolio_optimization).

In [1]:
import pybroker
from pybroker import ExecContext, Strategy, YFinance

pybroker.enable_data_source_cache("rebalancing")

## Equal Position Sizing

Suppose we want to rebalance a long-only portfolio at the beginning of every month to maintain an equal allocation for each stock. 

First, we write a helper function to detect when the current bar is the start of a new month:

In [2]:
def start_of_month(ctxs: dict[str, ExecContext]) -> bool:
    dt = tuple(ctxs.values())[0].dt
    if dt.month != pybroker.param("current_month"):
        pybroker.param("current_month", dt.month)
        return True
    return False

Next, we write a `rebalance` function to set an equal target allocation for each asset at the beginning of every month:

In [3]:
def rebalance(ctxs: dict[str, ExecContext]):
    if start_of_month(ctxs):
        target = 1 / len(ctxs)
        for ctx in ctxs.values():
            ctx.set_target_shares(target, dir="long")

With the `rebalance` function complete, we can backtest our strategy using a portfolio of five stocks. To process all stocks simultaneously on each bar of data, we use the [Strategy.set_after_exec](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy.set_after_exec) method:

In [4]:
strategy = Strategy(YFinance(), start_date="1/1/2018", end_date="1/1/2023")
strategy.add_execution(None, ["TSLA", "NFLX", "AAPL", "NVDA", "AMZN"])
strategy.set_after_exec(rebalance)
result = strategy.backtest()

Backtesting: 2018-01-01 00:00:00 to 2023-01-01 00:00:00



Loading bar data...


[                       0%                       ]

[*******************   40%                       ]  2 of 5 completed

[**********************60%****                   ]  3 of 5 completed

...

 77% (981 of 1259) |################     | Elapsed Time: 0:00:00 ETA:   0:00:00

 86% (1091 of 1259) |#################   | Elapsed Time: 0:00:00 ETA:   0:00:00

 95% (1201 of 1259) |################### | Elapsed Time: 0:00:00 ETA:   0:00:00

100% (1259 of 1259) |####################| Elapsed Time: 0:00:00 Time:  0:00:00

Finished backtest: 0:00:01


The `set_after_exec` function runs after all executions added via `add_execution`. Since we passed `None` to `add_execution`, no execution logic runs prior to `after_exec`.

In [5]:
result.orders

,type,symbol,date,created,order_type,intent,shares,limit_price,market_price,fill_price,fees
id,,,,,,,,,,,
1,buy,AAPL,2018-01-03,2018-01-02,market,buy_to_open,464,NaN,43.31,43.31,0.0
2,buy,AMZN,2018-01-03,2018-01-02,market,buy_to_open,336,NaN,59.84,59.84,0.0
3,buy,NFLX,2018-01-03,2018-01-02,market,buy_to_open,994,NaN,20.39,20.39,0.0
4,buy,NVDA,2018-01-03,2018-01-02,market,buy_to_open,4013,NaN,5.22,5.22,0.0
5,buy,TSLA,2018-01-03,2018-01-02,market,buy_to_open,869,NaN,21.36,21.36,0.0
...,...,...,...,...,...,...,...,...,...,...,...
293,sell,NFLX,2022-12-02,2022-12-01,market,sell_to_close,153,NaN,31.60,31.60,0.0
294,sell,NVDA,2022-12-02,2022-12-01,market,sell_to_close,974,NaN,16.69,16.69,0.0
295,buy,AAPL,2022-12-02,2022-12-01,market,buy_to_open,27,NaN,146.82,146.82,0.0


## Portfolio Optimization

[Portfolio optimization](https://en.wikipedia.org/wiki/Portfolio_optimization) guides rebalancing to meet specific objectives, such as allocating stocks to minimize risk.

[Riskfolio-Lib](https://riskfolio-lib.readthedocs.io/) is a popular Python library for portfolio optimization. You can install it using `pip install riskfolio-lib`.

The following example demonstrates how to construct a minimum risk portfolio by minimizing the [Conditional Value at Risk (CVaR)](https://www.investopedia.com/terms/c/conditional_value_at_risk.asp) based on the past year of returns:

In [6]:
import pandas as pd
import riskfolio as rp

pybroker.param("lookback", 252)  # Use past year of returns.


def calculate_returns(ctxs: dict[str, ExecContext], lookback: int):
    prices = {}
    for ctx in ctxs.values():
        prices[ctx.symbol] = ctx.adj_close[-lookback:]
    df = pd.DataFrame(prices)
    return df.pct_change().dropna()


def optimization(ctxs: dict[str, ExecContext]):
    lookback = pybroker.param("lookback")
    if start_of_month(ctxs):
        Y = calculate_returns(ctxs, lookback)
        port = rp.Portfolio(returns=Y)
        port.assets_stats(method_mu="hist", method_cov="hist")
        w = port.optimization(
            model="Classic",
            rm="CVaR",
            obj="MinRisk",
            rf=0,  # Risk free rate.
            l=0,  # Risk aversion factor.
            hist=True,  # Use historical scenarios.
        )
        for symbol, ctx in ctxs.items():
            target = w.T[symbol].values[0]
            ctx.set_target_shares(target, dir="long")

For more information and examples, see the [official Riskfolio-Lib documentation](https://riskfolio-lib.readthedocs.io/). Next, we backtest the strategy:

In [7]:
strategy.set_after_exec(optimization)
result = strategy.backtest(warmup=pybroker.param("lookback"))

Backtesting: 2018-01-01 00:00:00 to 2023-01-01 00:00:00



Loaded cached bar data.



Test split: 2018-01-02 00:00:00 to 2022-12-30 00:00:00


  0% (0 of 1259) |                       | Elapsed Time: 0:00:00 ETA:  --:--:--

 23% (291 of 1259) |####                 | Elapsed Time: 0:00:00 ETA:   0:00:00

...

 89% (1121 of 1259) |#################   | Elapsed Time: 0:00:01 ETA:   0:00:00

 93% (1181 of 1259) |##################  | Elapsed Time: 0:00:01 ETA:   0:00:00

 99% (1251 of 1259) |################### | Elapsed Time: 0:00:01 ETA:   0:00:00

100% (1259 of 1259) |####################| Elapsed Time: 0:00:01 Time:  0:00:01

Finished backtest: 0:00:01


In [8]:
result.orders.head()

,type,symbol,date,created,order_type,intent,shares,limit_price,market_price,fill_price,fees
id,,,,,,,,,,,
1,buy,AAPL,2019-01-04,2019-01-03,market,buy_to_open,1420,NaN,36.54,36.54,0.0
2,buy,AMZN,2019-01-04,2019-01-03,market,buy_to_open,347,NaN,77.81,77.81,0.0
3,buy,TSLA,2019-01-04,2019-01-03,market,buy_to_open,1020,NaN,20.69,20.69,0.0
4,sell,AAPL,2019-02-04,2019-02-01,market,sell_to_close,103,NaN,42.37,42.37,0.0
5,buy,AMZN,2019-02-04,2019-02-01,market,buy_to_open,1,NaN,81.58,81.58,0.0


The portfolio optimization allocated the entire portfolio to `AAPL`, `AMZN`, and `TSLA` during the first month of the backtest.